# Lab 1: Customer Support Agent using Strands Agents

## Overview

[Strands Agents](https://strandsagents.com/latest/) is a simple yet powerful SDK that takes a model-driven approach to building and running AI agents.

In this lab, you'll create a **Customer Support Agent** using the Strands Agents framework. We will use this Agent to explore the capabilities provided by Amazon Bedrock AgentCore in the following Labs.

The **Customer Support Agent** has the following tools available:
- **get_shipping_info()** -  Get shipping information of an order
- **get_return_policy()** - Get return policy for specific products
- **get_product_info()** - Get product information
- **get_order_status()** - Get order status

You will also add Bedrock Guardrails to your agent to block any unwanted topics.

## Architecture

![Architecture Diagram](images/architecture_lab1_strands.png)

## Prerequisites

* Python 3.12+
* AWS credentials configured  
* Anthropic Claude 3.7 enabled on [Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
* Strands Agents and supporting libraries


## Step 1: Install Dependencies and Import Libraries

In [ ]:
# Install required packages
%pip install strands-agents boto3 -q

In [ ]:
# Import libraries
from strands.tools import tool
from scripts.utils import put_ssm_parameter

In [ ]:
# Get boto session
import boto3
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

## Step 2: Implementing custom tools

Next, we will implement the 4 mock tools which will be provided to the Customer Support Agent.

Defining tools in Strands Agent is extremely simple, just add a `@tool` decorator to your function, and provide a description of the tool in the function's docstring. Strands Agents will use the function documentation, typing and arguments to provide context on this tool to your agent. 

### Tool 1: Get Shipping Info

# Purpose: This tool allows customers to track their orders by providing detailed shipping information including delivery method, costs, tracking numbers, and current status. It handles orders at any stage - from preparation through delivery.


In [ ]:
@tool
def get_shipping_info(order_id: str) -> str:
    """
    Get shipping information for a specific order.

    Args:
        order_id: The order ID to look up shipping information for

    Returns:
        Formatted string with shipping details including method, cost, and status
    """
    # Mock shipping database - in real implementation, this would query a shipping system

    shipping_info = {
        "12345": {
            "method": "Standard Shipping",
            "estimated_delivery": "3-5 business days",
            "cost": "Free",
            "status": "Preparing for shipment"
        },
        "67890": {
            "method": "Express Shipping",
            "estimated_delivery": "1-2 business days",
            "cost": "$9.99",
            "tracking_number": "TRK123456789",
            "status": "In transit"
        },
        "11111": {
            "method": "Priority Shipping",
            "delivery_date": "Delivered on Jan 14, 2024",
            "cost": "$14.99",
            "status": "Delivered"
        }
    }

    shipping = shipping_info.get(order_id)
    if not shipping:
        return f"I couldn't find shipping information for order #{order_id}. This might be because the order hasn't shipped yet or the order number is incorrect."

    if shipping["status"] == "Delivered":
        return f"Order #{order_id} was delivered using {shipping['method']} (${shipping['cost']}). {shipping['delivery_date']}."
    elif "tracking_number" in shipping:
        return f"Order #{order_id} is being shipped via {shipping['method']} (${shipping['cost']}). " \
               f"Tracking number: {shipping['tracking_number']}. " \
               f"Expected delivery: {shipping['estimated_delivery']}. Status: {shipping['status']}."
    else:
        return f"Order #{order_id} will be shipped using {shipping['method']} (${shipping['cost']}). " \
               f"Estimated delivery: {shipping['estimated_delivery']}. Current status: {shipping['status']}."


### Tool 2: Get Return Policy

# Purpose: This tool helps customers understand return policies for different product categories. It provides detailed information about return windows, conditions, processes, and refund timelines so customers know exactly what to expect when returning items.

In [ ]:
@tool
def get_return_policy(product_category: str) -> str:
    """
    Get return policy information for a specific product category.

    Args:
        product_category: The category of product (e.g., 'electronics', 'clothing', 'books')

    Returns:
        Formatted return policy details including timeframes and conditions
    """
    # Mock return policy database - in real implementation, this would query policy database
    return_policies = {
        "electronics": {
            "window": "30 days",
            "condition": "Items must be in original packaging with all accessories",
            "process": "Contact customer service to initiate return",
            "refund_time": "5-7 business days after we receive the item",
            "shipping": "Free return shipping on defective items"
        },
        "clothing": {
            "window": "60 days",
            "condition": "Items must be unworn, unwashed, and have tags attached",
            "process": "Use our online return portal or contact customer service",
            "refund_time": "3-5 business days after we receive the item",
            "shipping": "Customer pays return shipping unless item is defective"
        },
        "books": {
            "window": "14 days",
            "condition": "Books must be in original condition with no writing or damage",
            "process": "Contact customer service for return authorization",
            "refund_time": "3-5 business days after we receive the item",
            "shipping": "Customer pays return shipping"
        }
    }

    # Default policy for unlisted categories
    default_policy = {
        "window": "30 days",
        "condition": "Items must be in original condition and packaging",
        "process": "Contact customer service to initiate return",
        "refund_time": "5-7 business days after we receive the item",
        "shipping": "Return shipping policies vary by item"
    }

    policy = return_policies.get(product_category.lower(), default_policy)

    return f"Return policy for {product_category}:\\n\\n" \
           f"• Return window: {policy['window']} from delivery date\\n" \
           f"• Condition requirements: {policy['condition']}\\n" \
           f"• Return process: {policy['process']}\\n" \
           f"• Refund timeline: {policy['refund_time']}\\n" \
           f"• Return shipping: {policy['shipping']}"

### Tool 3: Get Product Information

# Purpose: This tool provides customers with comprehensive product details including warranties, available models, key features, shipping policies, and return information. It helps customers make informed purchasing decisions and understand what they're buying.

In [ ]:
@tool
def get_product_info(product_type: str) -> str:
    """
    Get detailed information about a specific product type.

    Args:
        product_type: The type of product to get information about

    Returns:
        Formatted product information including warranty, features, and policies
    """
    # Mock product catalog - in real implementation, this would query a product database
    products = {
        "laptops": {
            "warranty": "2-year comprehensive warranty",
            "models": "Available in 13-inch and 15-inch models",
            "features": "High-performance processors, SSD storage, premium displays",
            "shipping": "Free shipping on all orders",
            "return_policy": "30-day return policy"
        },
        "phones": {
            "warranty": "1-year manufacturer warranty",
            "models": "Multiple models available with various storage options",
            "features": "Latest cameras, 5G connectivity, long battery life",
            "shipping": "Free shipping on orders over $50",
            "return_policy": "14-day return policy"
        },
        "tablets": {
            "warranty": "1-year warranty with optional extended coverage",
            "models": "Available in 10-inch and 12-inch sizes",
            "features": "Touch screens, stylus support, lightweight design",
            "shipping": "Free shipping on all orders",
            "return_policy": "30-day return policy"
        }
    }

    product = products.get(product_type.lower())
    if not product:
        return f"I don't have specific information about {product_type}. Let me connect you with a specialist who can help with detailed product information."

    return f"Here's what I can tell you about our {product_type}:\\n\\n" \
           f"• Warranty: {product['warranty']}\\n" \
           f"• Models: {product['models']}\\n" \
           f"• Features: {product['features']}\\n" \
           f"• Shipping: {product['shipping']}\\n" \
           f"• Returns: {product['return_policy']}"



### Tool 4: Get Order Status

# Purpose: This tool allows customers to check the current status of their orders at any point in the fulfillment process. Whether an order is being processed, shipped, delivered, or returned, customers get relevant status information and next steps.

In [ ]:
@tool
def get_order_status(order_id: str) -> str:
    """
    Get the current status of a customer order.

    Args:
        order_id: The order ID to check status for

    Returns:
        Formatted order status information with relevant details
    """
    # Mock order database - in real implementation, this would query a database
    orders = {
        "12345": {
            "status": "processing",
            "date_ordered": "2024-01-15",
            "estimated_delivery": "2-3 business days"
        },
        "67890": {
            "status": "shipped",
            "date_shipped": "2024-01-16",
            "tracking_number": "TRK123456789",
            "estimated_delivery": "Tomorrow"
        },
        "11111": {
            "status": "delivered",
            "delivery_date": "2024-01-14",
            "delivered_to": "Front door"
        },
        "22222": {
            "status": "returned",
            "return_date": "2024-01-10",
            "refund_status": "Processed"
        }
    }

    order_info = orders.get(order_id)
    if not order_info:
        return f"I couldn't find order #{order_id} in our system. Please check the order number and try again."

    status = order_info["status"]
    if status == "processing":
        return f"Order #{order_id} is currently being processed. It was placed on {order_info['date_ordered']} and will ship within {order_info['estimated_delivery']}."
    elif status == "shipped":
        return f"Great news! Order #{order_id} was shipped on {order_info['date_shipped']}. Your tracking number is {order_info['tracking_number']} and it should arrive {order_info['estimated_delivery']}."
    elif status == "delivered":
        return f"Order #{order_id} was successfully delivered on {order_info['delivery_date']} to your {order_info['delivered_to']}."
    elif status == "returned":
        return f"Order #{order_id} was returned on {order_info['return_date']}. Your refund has been {order_info['refund_status'].lower()}."

    return f"Order #{order_id} status: {status}"

## Step 3: Create Bedrock Guardrails 

For illustrating the use of guardarails with our customer support assistant, we'll create a guardrail to simulate topics we don't want our chatbot to respond to, notably on finance - any question or instruction related to financial information, transactions, or related.

In [ ]:
from lab_helpers.lab1_guardrails import create_or_get_guardrail_resource

guardrail_id, guardrail_version = create_or_get_guardrail_resource()

In [ ]:
print("The guardrailId is", guardrail_id)
print("The guardrail version is", guardrail_version)

## Step 4: Create and Configure the Customer Support 

Next, we will create the Customer Support Agent providing a model, the list of tools implemented in the previous step, a system prompt, and the created Bedrock Guardrail resource.


In [ ]:
from strands import Agent
from strands.models import BedrockModel

SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For return policy questions
2. get_shipping_info() - Get shipping information for a specific order
3. get_order_status() - To get information about a the status of a specific order
4. get_product_info() - To get information about a specific product

Always use the appropriate tool to get accurate, up-to-date information rather than guessing."""

# Initialize the Bedrock model (Anthropic Claude 3.7 Sonnet)
model = BedrockModel(
    model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
    temperature=0.3,
    region_name=region,
    guardrail_id=guardrail_id,
    # guardrail_version=guardrail_version,
    guardrail_trace="enabled",  # Enable trace info for debugging
)

# Create the customer support agent with all 5 tools
agent = Agent(
    model=model,
    tools=[
        get_order_status,  # Tool 1: Simple order status lookup
        get_product_info,  # Tool 2: Simple product information lookup
        get_shipping_info,  # Tool 3: Simple shipping information lookup
        get_return_policy,  # Tool 4: Simple return policy lookup
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("Customer Support Agent created successfully!")

## Step 5: Test the Customer Support Agent

Let's test our agent with sample queries to ensure all tools work correctly.

### Test Return Policy Tool

In [ ]:
response = agent("What's the return policy for electronics?")

### Test Order Status Tool

In [ ]:
response = agent("Can you tell me the status of my order 12345?")

### Test Bedrock Guardrail

In [ ]:
response = agent("Where can I invest now?")

## 🎉 Congratulations!

You've successfully created your first Strands agent secured with Bedrock Guardrails.
